### Validação dos identificadores dos clientes

A coluna `customerID` identifica cada cliente e deve conter um valor
preenchido e único por registro.

A validação realiza as seguintes verificações:

1. Remove espaços nas extremidades dos identificadores em uma série auxiliar.
2. Considera valores vazios ou compostos apenas por espaços como ausentes.
3. Conta os identificadores distintos e verifica se existem repetições.
4. Exibe os registros com problemas e interrompe a célula com um erro caso
   existam identificadores ausentes ou duplicados.

O parâmetro `keep=False` marca todas as ocorrências de um identificador
repetido, incluindo a primeira. O CSV original permanece intacto.

In [1]:
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
import numpy as np

In [2]:
# Procura o .env no diretório atual e nas pastas superiores.
arquivo_env = find_dotenv(usecwd=True)

if not arquivo_env:
    raise FileNotFoundError("Arquivo .env não encontrado.")

load_dotenv(arquivo_env)

# Usa a pasta que contém o .env como raiz do projeto.
projeto = Path(arquivo_env).parent

caminho_raw = projeto / os.environ["RAW_CSV_PATH"]
caminho_processed = projeto / os.environ["PROCESSED_CSV_PATH"]

df_raw = pd.read_csv(caminho_raw, dtype={"customerID": "string"})

if "customerID" not in df_raw.columns:
    raise ValueError("A coluna 'customerID' não foi encontrada na base raw.")

# Normaliza apenas uma série auxiliar para realizar a conferência.
ids = df_raw["customerID"].str.strip().replace("", pd.NA)

ausentes = ids.isna()
duplicados = ids.notna() & ids.duplicated(keep=False)

print(f"Total de registros: {len(df_raw)}")
print(f"Identificadores distintos preenchidos: {ids.nunique()}")
print(f"Identificadores ausentes: {ausentes.sum()}")
print(f"Registros com identificadores repetidos: {duplicados.sum()}")

if df_raw.empty:
    raise ValueError("A base raw está vazia.")

if ausentes.any() or duplicados.any():
    display(df_raw.loc[ausentes | duplicados])
    raise ValueError(
        "Validação reprovada: existem identificadores ausentes ou duplicados."
    )

print("Validação aprovada: todos os identificadores estão preenchidos e são únicos.")

NameError: name 'pd' is not defined

### Validação das colunas esperadas

Esta verificação compara as colunas da base raw com as 21 colunas
previstas para o conjunto de dados.

A validação identifica colunas ausentes, inesperadas ou com nomes
duplicados. Caso encontre algum desses problemas, apresenta os detalhes
e gera um erro.

A ordem das colunas não interfere na validação.

In [ ]:
colunas_esperadas = {
    "customerID",
    "gender",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "tenure",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "Churn",
}

colunas_encontradas = set(df_raw.columns)

colunas_ausentes = sorted(colunas_esperadas - colunas_encontradas)
colunas_inesperadas = sorted(colunas_encontradas - colunas_esperadas)
colunas_duplicadas = df_raw.columns[
    df_raw.columns.duplicated()
].tolist()

if colunas_ausentes or colunas_inesperadas or colunas_duplicadas:
    raise ValueError(
        "Validação das colunas reprovada.\n"
        f"Colunas ausentes: {colunas_ausentes}\n"
        f"Colunas inesperadas: {colunas_inesperadas}\n"
        f"Colunas duplicadas: {colunas_duplicadas}"
    )

print(
    f"Validação aprovada: a base contém exatamente "
    f"as {len(colunas_esperadas)} colunas esperadas."
)

Validação aprovada: a base contém exatamente as 21 colunas esperadas.


### Verificação de valores ausentes e consistência dos dados

A verificação identifica valores nulos, campos vazios e campos contendo
apenas espaços, apresentando a quantidade e o percentual por coluna.

Também verifica:
- Categorias permitidas para cada variável.
- Valores numéricos válidos, finitos e não negativos.
- `SeniorCitizen` com valores 0 ou 1.
- `tenure` com quantidade inteira de meses.
- Coerência entre os serviços contratados e suas categorias de ausência.

`No internet service` e `No phone service` são categorias válidas.

Ausências em `customerID` e `Churn` são consideradas erros. Nas demais
colunas, são reportadas para posterior decisão de tratamento.

As verificações usam uma cópia auxiliar e não preenchem valores ausentes.
Caso sejam encontradas inconsistências, a célula apresenta os problemas
e gera um erro.

In [ ]:
if df_raw.empty:
    raise ValueError("A base raw está vazia.")

# Usa uma cópia auxiliar para reconhecer campos vazios e espaços.
base = df_raw.astype("string").apply(lambda coluna: coluna.str.strip())
base = base.replace("", pd.NA)

# 1. Relatório de valores ausentes.
resumo_ausentes = pd.DataFrame({
    "quantidade": base.isna().sum(),
    "percentual": (base.isna().mean() * 100).round(2),
}).rename_axis("coluna")

print("Valores ausentes por coluna:")
display(resumo_ausentes.loc[resumo_ausentes["quantidade"] > 0])

# 2. Categorias permitidas na base raw.
servicos_internet = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies",
]

categorias_permitidas = {
    "gender": {"Female", "Male"},
    "Partner": {"Yes", "No"},
    "Dependents": {"Yes", "No"},
    "PhoneService": {"Yes", "No"},
    "MultipleLines": {"Yes", "No", "No phone service"},
    "InternetService": {"DSL", "Fiber optic", "No"},
    "Contract": {"Month-to-month", "One year", "Two year"},
    "PaperlessBilling": {"Yes", "No"},
    "PaymentMethod": {
        "Electronic check", "Mailed check",
        "Bank transfer (automatic)", "Credit card (automatic)",
    },
    "Churn": {"Yes", "No"},
}
categorias_permitidas.update({
    coluna: {"Yes", "No", "No internet service"}
    for coluna in servicos_internet
})

falhas = pd.DataFrame(index=base.index)

for coluna in ["customerID", "Churn"]:
    falhas[f"{coluna}: preenchimento obrigatório"] = base[coluna].isna()

for coluna, categorias in categorias_permitidas.items():
    falhas[f"{coluna}: categoria inválida"] = (
        base[coluna].notna() & ~base[coluna].isin(categorias)
    )

# 3. Validade dos valores numéricos.
colunas_numericas = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
numeros = base[colunas_numericas].apply(pd.to_numeric, errors="coerce")

for coluna in colunas_numericas:
    valores = numeros[coluna]
    falhas[f"{coluna}: número inválido ou negativo"] = (
        base[coluna].notna()
        & (valores.isna() | ~np.isfinite(valores) | valores.lt(0))
    )

falhas["SeniorCitizen: valor diferente de 0 ou 1"] = (
    numeros["SeniorCitizen"].notna()
    & ~numeros["SeniorCitizen"].isin([0, 1])
)

tenure_finito = numeros["tenure"].where(np.isfinite(numeros["tenure"]))
falhas["tenure: quantidade de meses fracionária"] = tenure_finito.mod(1).ne(0)

# 4. Coerência entre contratação e ausência dos serviços.
falhas["PhoneService incompatível com MultipleLines"] = (
    base["PhoneService"].notna()
    & base["MultipleLines"].notna()
    & base["PhoneService"].eq("No").ne(
        base["MultipleLines"].eq("No phone service")
    )
)

for coluna in servicos_internet:
    falhas[f"{coluna}: incompatível com InternetService"] = (
        base["InternetService"].notna()
        & base[coluna].notna()
        & base["InternetService"].eq("No").ne(
            base[coluna].eq("No internet service")
        )
    )

# 5. Resultado das verificações.
falhas = falhas.fillna(False)
resumo_falhas = falhas.sum()
resumo_falhas = resumo_falhas.loc[resumo_falhas > 0]

if not resumo_falhas.empty:
    display(resumo_falhas.to_frame("registros_com_falha"))
    display(df_raw.loc[falhas.any(axis=1)].head(20))
    raise ValueError(
        "Foram encontradas inconsistências. Consulte os resultados acima."
    )

if base.isna().any().any():
    print(
        "Verificação concluída com pendências: há valores ausentes "
        "que precisam ser avaliados. "
        "As demais verificações de consistência passaram."
    )
else:
    print(
        "Verificação concluída: não foram encontrados valores ausentes "
        "nem inconsistências nas regras avaliadas."
    )

Valores ausentes por coluna:


,quantidade,percentual
coluna,,
TotalCharges,11,0.16


Verificação concluída com pendências: há valores ausentes que precisam ser avaliados. As demais verificações de consistência passaram.
